In [4]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    filename="logs/as_243/mgmt_operations.as_243.renames.log",
    encoding="utf-8",
)
logging.getLogger('epicsarchiver').setLevel(logging.DEBUG)
LOG: logging.Logger = logging.getLogger(__name__)

In [5]:
from epicsarchiver.mgmt.archiver_mgmt_info import ArchivingStatus
from epicsarchiver.mgmt.archiver_mgmt_operations import Storage

In [6]:
def is_alias(archiver, old_pv, new_pv):
    """
    Check if old_pv is an alias of new_pv
    """
    old_pv_details: list[dict[str, str]] = archiver.get_pv_details(old_pv)
    aliases_entry = [entry for entry in old_pv_details if entry["name"] == "Alias for "]
    for entry in aliases_entry:
        if entry["value"] == new_pv:
            return True
    return False

In [7]:
import random
from epicsarchiver import ArchiverAppliance

def pause_multiple_pvs(archiver, pv_list):
    for pv in pv_list:
        LOG.info(f"Pausing PV: {pv}")
        pause_result = archiver.pause_pv(pv)
        LOG.info(f"Pause result: {pause_result}")

def check_rename_pv(archiver, old_pv, new_pv) -> True:
    if old_pv == new_pv:
        LOG.info(f"Skipping {old_pv} to {new_pv}, already the same")
        return False
    new_pv_status = archiver.get_archiving_status(new_pv)
    old_pv_status = archiver.get_archiving_status(new_pv)
    if new_pv_status == ArchivingStatus.NotBeingArchived or old_pv_status == ArchivingStatus.NotBeingArchived:
        LOG.info(f"Skipping {old_pv}: {old_pv_status} to {new_pv}: {new_pv_status}")
        return False
    if is_alias(archiver, old_pv, new_pv):
        LOG.info(f"Skipping {old_pv} to {new_pv}, already an alias")
        return False
    return True
    

def rename_pv(archiver, old_pv, new_pv) -> dict[str, dict[str, str]]:
    pause_multiple_pvs(archiver, [old_pv, new_pv])
    LOG.info(f"Renaming {old_pv} to {new_pv}")
    new_pv_status = archiver.get_archiving_status(new_pv)
    LOG.info(f"New PV status: {new_pv_status}")
    if new_pv_status == ArchivingStatus.Paused:
        archiver.rename_and_append(old_pv, new_pv, Storage.MTS)
        LOG.info(f"Renamed {old_pv} to {new_pv}")
    else:
        archiver.pause_rename_resume_pv(old_pv, new_pv)
        LOG.info(f"Renamed {old_pv} to {new_pv}")
    new_pv_status = archiver.get_archiving_status(new_pv)
    if new_pv_status == ArchivingStatus.Paused:
        LOG.info(f"Resuming {new_pv}")
        archiver.resume_pv(new_pv)
    else:
        return {new_pv: {"status": new_pv_status, "old_pv": old_pv}}
    return {}

def rename_pvs(archivers, renames: list[tuple[str, str]]):
    rename_results = {}
    for old_pv, new_pv in renames:
        archiver = random.choice(archivers)
        if check_rename_pv(archiver, old_pv, new_pv):
            rename_results[new_pv] = rename_pv(archiver, old_pv, new_pv)
        else:
            rename_results[new_pv] = {new_pv: {"status": "skipped", "old_pv": old_pv}}
    return rename_pvs


            

In [8]:
from concurrent.futures import ThreadPoolExecutor


def parallel_execute_rename(archivers, renames: list[tuple[str, str]]):
    def rename_pv_task(task_input):
        archivers, old_pv, new_pv = task_input
        archiver = ArchiverAppliance(random.choice(archivers).hostname)
        if check_rename_pv(archiver, old_pv, new_pv):
            return rename_pv(archiver, old_pv, new_pv)
        else:
            return {new_pv: {"status": "skipped", "old_pv": old_pv}}
         
    with ThreadPoolExecutor() as executer:
        return executer.map(rename_pv_task, [(archivers, old_pv, new_pv) for old_pv, new_pv in renames])

In [9]:
from pathlib import Path

hbl_files = [f for f in Path("AS-243").iterdir() if f.is_file() and f.suffix == ".csv"]

In [10]:
import csv

actions = ["Pause", "Delete", "Rename"]

def read_file(filename) -> dict[str, list[tuple[str, str]]]:
    """Reads a file, returns action, plus PVs needing actions"""
    output = []
    csv_data = csv.reader(open(filename), delimiter=';')
    for row in csv_data:
        if row[1] not in actions[0:2] and row[1]:
            output.append((row[0].split()[0], row[1].split()[0]))
    return output



In [11]:
hbl_data: dict[str, list[tuple[str, str]]] = {f.name[-7:-4]: read_file(f) for f in hbl_files}

In [12]:
archivers_tn = [ArchiverAppliance(f"archiver-linac-{n:02}.tn.esss.lu.se") for n in range(2, 9)]

In [13]:
# result_pvs_to_rename = parallel_execute_rename(archivers_tn, hbl_data["010"] + hbl_data["020"] + hbl_data["030"] + hbl_data["040"])

In [14]:
renames = hbl_data["010"] + hbl_data["020"] + hbl_data["030"] + hbl_data["040"]



In [16]:

def check_failed_rename(archiver, old_pv, new_pv) -> dict[str, str] | None:
    new_pv_status = archiver.get_archiving_status(new_pv)
    old_pv_status = archiver.get_archiving_status(new_pv)
    if new_pv_status in [ArchivingStatus.Paused, ArchivingStatus.NotBeingArchived] or old_pv_status in [ArchivingStatus.Paused, ArchivingStatus.NotBeingArchived]:
        return {new_pv: new_pv_status, old_pv: old_pv_status}
    return None

def find_prolematic_pvs(archivers, renames):
    def get_pvs(task_input):
        archivers, old_pv, new_pv = task_input
        archiver = ArchiverAppliance(random.choice(archivers).hostname)
        return check_failed_rename(archiver, old_pv, new_pv)
         
    with ThreadPoolExecutor() as executer:
        return executer.map(get_pvs, [(archivers, old_pv, new_pv) for old_pv, new_pv in renames])

def problem_process(result):
    for r in result:
        if r:
            LOG.info(r)
    return result

    

In [17]:
result = find_prolematic_pvs(archivers_tn, renames)

In [18]:
problem_process(result)

<generator object Executor.map.<locals>.result_iterator at 0x118d44d60>